# tensor-to-device — ex1: move tensor to chosen device with the CPU/CUDA guard

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `tensor-to-device`. Running the final beacon cell reports progress against the `PyTorch: tensor.to(device)` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: tensor.to(device)` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`tensor-to-device`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "tensor-to-device"
DD_SUBTOPIC = "PyTorch: tensor.to(device)"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `tensor.to(device)` — quick refresher

`x.to(device)` returns a new tensor on the target device. It is a **copy** when the device differs and a **no-op view** when the device already matches.

**Critical rules.**
- `x.to(device)` is **not in-place** — you must reassign: `x = x.to(device)`.
- All inputs to an op must live on the same device. Mixing CPU and CUDA tensors raises `RuntimeError`.
- `to()` also accepts a dtype: `x.to(dtype=t.float16)` or `x.to(device='cuda', dtype=t.float16)`.
- For modules, `model.to(device)` IS in-place (moves all parameters and buffers).

**Idiomatic guard.** Pick the device once at the top of the script and thread it through:
```python
device = 'cuda' if t.cuda.is_available() else 'cpu'
x = x.to(device)
model = model.to(device)
```

### Exercise 1 — move tensor to chosen device with the CPU/CUDA guard

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Pick a target device with the standard `cuda.is_available()` guard, move a tensor to it via `.to(device)` with reassignment, and confirm the move was not in place.
> Keywords: device, cpu, cuda, to, guard
> ```

**KCs targeted:** `pick-device-with-cuda-available`, `to-is-not-inplace`

Implement `ex1_to_best_device(x)`.

1. Pick `device = 'cuda' if t.cuda.is_available() else 'cpu'`.
2. Return `x.to(device)` — **without modifying `x` itself** (do NOT do `x = x.to(...)` and then `return x` if there's any way for the caller to see the unmodified original; the test captures `id(x)` before the call and checks the moved tensor is a different object when the device actually changed).
3. The function must work on a CPU-only machine — the test always runs on CPU, so `device` will resolve to `'cpu'` and the returned tensor's device should be `'cpu'`.

Input: `x` — any `torch.Tensor`.
Output: a `torch.Tensor` on the chosen device, with identical shape and dtype to `x`.

**Why this is its own drill.** Forgetting that `.to()` is not-in-place — and missing the reassignment — is the #1 'why is my model still on CPU' bug for newcomers.

In [ ]:
def ex1_to_best_device(x: Tensor) -> Tensor:
    """Pick device via cuda.is_available(), return x.to(device)."""
    raise NotImplementedError()


def _test_ex1():
    # CPU-only target environment — this test passes both on CPU and CUDA
    # boxes (the chosen device tracks t.cuda.is_available()).
    x = t.arange(12, dtype=t.float32).reshape(3, 4)
    x_orig = x.clone()  # for value comparison
    expected_device = t.device('cuda' if t.cuda.is_available() else 'cpu')

    out = ex1_to_best_device(x)

    # Shape + dtype preserved.
    assert out.shape == x.shape
    assert out.dtype == x.dtype
    # Device matches the cuda.is_available() guard.
    assert out.device.type == expected_device.type, (
        f'expected device {expected_device}, got {out.device}'
    )
    # Values intact (matches the original).
    out_cpu = out.cpu()
    assert t.equal(out_cpu, x_orig), 'value content changed during move'

    # Confirm the source tensor was not modified in place.
    assert t.equal(x, x_orig), 'source tensor was modified — .to() must not be in place'

    # When already on the target device, .to() should be a no-op view
    # (or the same tensor) — but at minimum return something equal.
    y = t.zeros(5, device='cpu')
    y_out = ex1_to_best_device(y)
    assert y_out.device.type == 'cpu' if not t.cuda.is_available() else True
    assert t.equal(y_out.cpu(), y.cpu())
    print(f'tensor moved to {out.device} (cuda available: {t.cuda.is_available()})')
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_to_best_device(x: Tensor) -> Tensor:
    device = 'cuda' if t.cuda.is_available() else 'cpu'
    return x.to(device)
```

**`.to()` is not in-place.** It returns a (potentially) new tensor. You MUST capture the return value with `out = x.to(device)` or `x = x.to(device)`. Code that just writes `x.to(device)` and moves on silently leaves `x` on the original device.

**Contrast with `model.to(device)`.** Modules DO move in place — `model.to(device)` mutates the parameters / buffers directly. Tensors don't. This asymmetry catches a LOT of PyTorch newcomers.

**The `cuda.is_available()` guard is the universal pattern.** Hard-coding `device='cuda'` breaks every CI run that doesn't have a GPU. Putting the guard at the top of the script and threading the chosen device everywhere downstream keeps the code portable.

**Same-device `to()` may return the same object.** No copy is made if `x` is already on `device` with matching dtype. Don't rely on identity (`is`) either way — just trust the device attribute.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()